# 06 — Presentation figures (segmentation, uncertainty, review flag, failure case)

Generates every figure the BTP deck needs from the **trained checkpoints already in `models/`** and the
**official val split (= our test set)**. Runs on CPU in a few minutes; no training.

| File (in `results/figures/`) | Deck slide | Model used |
|---|---|---|
| `fig_seg_examples.png` | Segmentation | headline U-Net (seed 0) |
| `fig_unet_vs_deeplab_silkspot.png` | Segmentation by class | U-Net vs DeepLabV3+ (seed 0) |
| `fig_uncertainty_mixed.png` | Uncertainty | dropout U-Net, 30 MC passes |
| `fig_review_flag.png` | Uncertainty / Deployment | dropout U-Net |
| `fig_failure_confident_wrong_class.png` | Limitations | dropout U-Net + LLM-eval records |
| `fig_reliability.png`, `fig_per_class_iou.png` | Uncertainty / Segmentation by class | result JSONs only |

**Honest example selection:** for each class, a fixed random sample (seed 0) of test images is scored and
the image whose per-image IoU is **closest to the sample median** is shown — representative, not cherry-picked.

Images are fetched one file at a time with `hf_hub_download` (same as notebook 01), so the full dataset is not
needed. If you have a local clone, set `DATA_ROOT` to the folder holding `val/`, `val_mask/` and `val-text.json`.

In [ ]:
import os, sys, json, random
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():          # notebook opened from notebooks/
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import numpy as np, cv2, torch
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from src.segmentation.model import build_model, enable_mc_dropout
from src.config import REVIEW_CONFIDENCE_THRESHOLD

DATA_ROOT = os.environ.get("SDX_DATA_ROOT")   # or e.g. r"C:\data\SteelDefectX"; None -> download from HF
REPO_ID = "Zhaosxian/SteelDefectX"
OUT = ROOT / "results" / "figures"; OUT.mkdir(parents=True, exist_ok=True)
N_PER_CLASS = 30        # test images scored per class when choosing a representative example
MC_PASSES = 30          # same as notebook 03
SEED = 0
UNKNOWN_THR = json.load(open(ROOT / "models" / "class_threshold.json"))["threshold"]

device = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

INK, ORANGE, BLUE, PAPER = "#16202A", "#E8833A", "#3E6A8A", "#F4F3EF"
plt.rcParams.update({"font.size": 11, "axes.titlesize": 11.5, "figure.facecolor": "white",
                     "savefig.facecolor": "white", "savefig.bbox": "tight", "savefig.dpi": 200})
print("device:", device, "| review threshold:", REVIEW_CONFIDENCE_THRESHOLD, "| unknown-class threshold:", UNKNOWN_THR)

## 1. Data and models

In [ ]:
def _fetch(rel):
    if DATA_ROOT:
        return str(Path(DATA_ROOT) / rel)
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=REPO_ID, filename=rel, repo_type="dataset")

val_text = json.load(open(_fetch("val-text.json"), encoding="utf-8"))
test_by_class = {}
for e in val_text:
    test_by_class.setdefault(e["class_name"], []).append(e["image_name"])
name_to_class = {e["image_name"]: e["class_name"] for e in val_text}
print(len(val_text), "test images,", len(test_by_class), "classes")

def load_pair(name):
    # same preprocessing as src/data/dataset.py: grayscale, 256x256, mask > 127
    img = cv2.imread(_fetch(f"val/{name}"), cv2.IMREAD_GRAYSCALE)
    msk = cv2.imread(_fetch(f"val_mask/{Path(name).stem}.png"), cv2.IMREAD_GRAYSCALE)
    if img.shape != (256, 256):
        img = cv2.resize(img, (256, 256), interpolation=cv2.INTER_LINEAR)
    if msk.shape != (256, 256):
        msk = cv2.resize(msk, (256, 256), interpolation=cv2.INTER_NEAREST)
    return img, (msk > 127).astype(np.uint8)

In [ ]:
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)[:, None, None]
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)[:, None, None]

def to_tensor(gray):
    x = np.stack([gray] * 3, 0).astype(np.float32) / 255.0
    return torch.from_numpy((x - MEAN) / STD)[None].to(device)

def load_model(arch, ckpt, dropout=0.0):
    m = build_model(arch, "resnet34", None, dropout=dropout)
    m.load_state_dict(torch.load(ROOT / ckpt, map_location=device))
    return m.to(device).eval()

unet = load_model("unet", "models/segmentation/unet_resnet34_seed0.pt")
deeplab = load_model("deeplabv3plus", "models/segmentation/deeplabv3plus_resnet34_seed0.pt")
unet_mc = load_model("unet", "models/unet_dropout.pt", dropout=0.2)   # the model the dashboard runs

@torch.no_grad()
def predict(model, gray):
    model.eval()
    return torch.sigmoid(model(to_tensor(gray)))[0, 0].cpu().numpy()

@torch.no_grad()
def predict_mc(gray, n=MC_PASSES):
    # identical rule to dashboard predict(): confidence = mean prob over predicted-defect pixels
    enable_mc_dropout(unet_mc)
    x = to_tensor(gray)
    ps = torch.stack([torch.sigmoid(unet_mc(x)) for _ in range(n)])[:, 0, 0].cpu().numpy()
    prob, unc = ps.mean(0), ps.std(0, ddof=1)
    conf = float(prob[prob > 0.5].mean()) if (prob > 0.5).any() else float(1 - prob.mean())
    return prob, unc, conf

def iou(prob, gt):
    p, g = prob > 0.5, gt.astype(bool)
    union = (p | g).sum()
    return 1.0 if union == 0 else float((p & g).sum() / union)

In [ ]:
_scored = {}
def scored(cls, n=N_PER_CLASS):
    # U-Net + DeepLabV3+ on a fixed random sample of this class's test images, sorted by U-Net IoU
    if cls not in _scored:
        names = random.Random(SEED).sample(sorted(test_by_class[cls]), min(n, len(test_by_class[cls])))
        rows = []
        for nm in names:
            img, gt = load_pair(nm)
            pu, pd = predict(unet, img), predict(deeplab, img)
            rows.append(dict(name=nm, cls=cls, img=img, gt=gt, p_unet=pu, p_dl=pd,
                             iou_unet=iou(pu, gt), iou_dl=iou(pd, gt)))
        _scored[cls] = sorted(rows, key=lambda r: r["iou_unet"])
    return _scored[cls]

def nearest_to_median(rows, value):
    vals = np.array([value(r) for r in rows])
    return rows[int(np.argmin(np.abs(vals - np.median(vals))))]

def representative(cls):
    return nearest_to_median(scored(cls), lambda r: r["iou_unet"])

def error_overlay(img, prob, gt):
    p, g = prob > 0.5, gt.astype(bool)
    rgb = np.stack([img] * 3, -1).astype(np.float32) / 255.0 * 0.8
    for m, hexcol, a in ((p & g, PAPER, 0.55), (p & ~g, ORANGE, 0.85), (~p & g, BLUE, 0.85)):
        c = np.array([int(hexcol[i:i + 2], 16) / 255 for i in (1, 3, 5)])
        rgb[m] = (1 - a) * rgb[m] + a * c
    return rgb

ERR_LEGEND = [Patch(facecolor=PAPER, edgecolor="#9AA4AD", label="correctly segmented"),
              Patch(facecolor=ORANGE, label="false positive"),
              Patch(facecolor=BLUE, label="missed (false negative)")]

def show(ax, im, title, kind="img"):
    if kind == "img":
        ax.imshow(im, cmap="gray", vmin=0, vmax=255)
    elif kind == "mask":
        ax.imshow(im, cmap="gray", vmin=0, vmax=1)
    else:
        ax.imshow(im)
    ax.set_title(title); ax.axis("off")

## 2. Segmentation examples: image | ground truth | prediction | errors

In [ ]:
SEG_ROWS = [("Punching", "highest-IoU class"), ("Welding line", "typical class"),
            ("Rolled pit", "rare class"), ("Waist folding", "lowest-IoU class")]

fig, ax = plt.subplots(len(SEG_ROWS), 4, figsize=(12, 3.15 * len(SEG_ROWS)))
for r, (cls, tag) in enumerate(SEG_ROWS):
    ex = representative(cls)
    show(ax[r, 0], ex["img"], f"{cls} ({tag})")
    show(ax[r, 1], ex["gt"], "Ground truth", "mask")
    show(ax[r, 2], ex["p_unet"] > 0.5, "U-Net prediction", "mask")
    show(ax[r, 3], error_overlay(ex["img"], ex["p_unet"], ex["gt"]), f"Errors · image IoU {ex['iou_unet']:.2f}", "rgb")
    print(f"{cls:15s} {ex['name']:16s} IoU={ex['iou_unet']:.3f}  (sample median of {len(scored(cls))})")
fig.legend(handles=ERR_LEGEND, loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.01))
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig(OUT / "fig_seg_examples.png"); plt.show()

## 3. U-Net vs DeepLabV3+ on Silk spot (thin, banded defect)

In [ ]:
rows = scored("Silk spot")
gap = lambda r: r["iou_unet"] - r["iou_dl"]
picks = [("median gap in sample", nearest_to_median(rows, gap)),
         ("largest gap in sample", max(rows, key=gap))]
print(f"Silk spot sample (n={len(rows)}): mean image IoU  U-Net {np.mean([r['iou_unet'] for r in rows]):.3f}"
      f"  DeepLabV3+ {np.mean([r['iou_dl'] for r in rows]):.3f}")

fig, ax = plt.subplots(len(picks), 4, figsize=(12, 3.15 * len(picks)))
for r, (tag, ex) in enumerate(picks):
    show(ax[r, 0], ex["img"], f"Silk spot ({tag})")
    show(ax[r, 1], ex["gt"], "Ground truth", "mask")
    show(ax[r, 2], error_overlay(ex["img"], ex["p_unet"], ex["gt"]), f"U-Net · IoU {ex['iou_unet']:.2f}", "rgb")
    show(ax[r, 3], error_overlay(ex["img"], ex["p_dl"], ex["gt"]), f"DeepLabV3+ · IoU {ex['iou_dl']:.2f}", "rgb")
fig.legend(handles=ERR_LEGEND, loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.savefig(OUT / "fig_unet_vs_deeplab_silkspot.png"); plt.show()

## 4. MC-Dropout uncertainty across different classes (replaces the all-Bright-scratch figure)

In [ ]:
UNC_CLASSES = ["Bright scratch", "Inclusion", "Rolled in scale", "Crazing"]

results = []
for cls in UNC_CLASSES:
    ex = representative(cls)
    prob, unc, conf = predict_mc(ex["img"])
    results.append((ex, prob, unc, conf))
vmax = max(float(u.max()) for _, _, u, _ in results)

fig, ax = plt.subplots(len(results), 4, figsize=(13, 3.4 * len(results)), layout="constrained")
for r, (ex, prob, unc, conf) in enumerate(results):
    status = "passes" if conf >= REVIEW_CONFIDENCE_THRESHOLD else "flagged for review"
    show(ax[r, 0], ex["img"], ex["cls"])
    show(ax[r, 1], ex["gt"], "Ground truth", "mask")
    ax[r, 2].imshow(prob, cmap="viridis", vmin=0, vmax=1); ax[r, 2].set_title("Mean probability"); ax[r, 2].axis("off")
    im = ax[r, 3].imshow(unc, cmap="inferno", vmin=0, vmax=vmax)
    ax[r, 3].set_title(f"Uncertainty\nseg conf {conf:.2f} · {status}"); ax[r, 3].axis("off")
    print(f"{ex['cls']:16s} {ex['name']:14s} seg confidence={conf:.3f}  mean unc={unc.mean():.4f}")
fig.colorbar(im, ax=ax[:, 3], shrink=0.6, label="std of probability across 30 passes")
plt.savefig(OUT / "fig_uncertainty_mixed.png"); plt.show()

## 5. Review flag: one report that passes, one that is flagged (from the LLM-eval records)

In [ ]:
diag = [json.loads(l) for l in open(ROOT / "results" / "llm" / "diagnoses.jsonl", encoding="utf-8")]
diag_by_name = {d["image_name"]: d for d in diag}
reports = [d for d in diag if not d["unknown"]]
passed = nearest_to_median([d for d in reports if not d["review_flag"]], lambda d: d["confidence"])
flagged = nearest_to_median([d for d in reports if d["review_flag"]], lambda d: d["confidence"])

fig, ax = plt.subplots(2, 3, figsize=(10, 6.6))
for r, d in enumerate([passed, flagged]):
    img, gt = load_pair(d["image_name"])
    prob, unc, conf = predict_mc(img)
    status = "flagged for manual review" if d["review_flag"] else "passes"
    show(ax[r, 0], img, f"{d['class_name']} · {d['image_name']}")
    show(ax[r, 1], error_overlay(img, prob, gt), f"Prediction · conf {d['confidence']:.2f}", "rgb")
    ax[r, 2].imshow(unc, cmap="inferno", vmin=0); ax[r, 2].set_title(f"Uncertainty · {status}"); ax[r, 2].axis("off")
    print(f"{d['image_name']}: stored conf {d['confidence']:.3f} (used in report), recomputed now {conf:.3f}")
fig.legend(handles=ERR_LEGEND, loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.savefig(OUT / "fig_review_flag.png"); plt.show()
print(f"\nThreshold {REVIEW_CONFIDENCE_THRESHOLD:.0%}. Use these two records for the JSON shown on the LLM slide:")
print(json.dumps({k: passed[k] for k in ("image_name", "pred_class", "cls_conf", "confidence", "attributes")}, indent=1))
print(json.dumps({k: v for k, v in passed["diag"].items() if not k.startswith("_")}, indent=1))

## 6. Failure case: a confident wrong class passes the unknown-class check

In [ ]:
wrong = [d for d in diag if not d["unknown"] and d["pred_class"] != d["class_name"]]
print(len(wrong), "of", len(reports), "reports used a wrong class:")
for d in wrong:
    print(f"  {d['image_name']}: true {d['class_name']}, predicted {d['pred_class']} (p={d['cls_conf']:.2f})")

if wrong:
    fig, ax = plt.subplots(len(wrong), 3, figsize=(10, 3.3 * len(wrong)), squeeze=False)
    for r, d in enumerate(wrong):
        img, gt = load_pair(d["image_name"])
        prob, _, _ = predict_mc(img)
        show(ax[r, 0], img, f"True class: {d['class_name']}")
        show(ax[r, 1], error_overlay(img, prob, gt), f"U-Net mask · seg conf {d['confidence']:.2f}", "rgb")
        ax[r, 2].axis("off")
        ax[r, 2].text(0, 0.5, f"Predicted: {d['pred_class']}\nclass p = {d['cls_conf']:.2f} ≥ {UNKNOWN_THR:.2f}\n"
                      f"→ passes the unknown-class check\n→ report uses the {d['pred_class']} cause",
                      fontsize=12, va="center", color=INK)
    plt.tight_layout()
    plt.savefig(OUT / "fig_failure_confident_wrong_class.png"); plt.show()

## 7. Figures from result JSONs only (no model needed)

In [ ]:
def bins_of(f):
    c = json.load(open(ROOT / "results" / "uncertainty" / f))
    b = [x for x in c["bins"] if x["weight"] > 0]
    return c["ECE"], np.array([x["conf"] for x in b]), np.array([x["acc"] for x in b]), np.array([x["weight"] for x in b])

ece1, conf1, acc1, w1 = bins_of("calibration.json")
ece2, conf2, acc2, w2 = bins_of("calibration_mc.json")
fig, (a, b) = plt.subplots(2, 1, figsize=(5.6, 6.6), gridspec_kw={"height_ratios": [3, 1.1]}, sharex=True)
a.plot([0, 1], [0, 1], "--", color="#9AA4AD", label="perfect calibration")
a.plot(conf1, acc1, "o-", color=BLUE, label=f"single pass (ECE {ece1:.2f}%)")
a.plot(conf2, acc2, "s-", color=ORANGE, label=f"MC-Dropout, 30 passes (ECE {ece2:.2f}%)")
a.set_ylabel("observed defect fraction"); a.set_title("Pixel reliability diagram, test set"); a.legend(frameon=False, fontsize=9.5)
a.set_xlim(0, 1); a.set_ylim(0, 1); a.grid(alpha=0.25)
b.bar(conf1, w1 / w1.sum() * 100, width=0.05, color=BLUE)
b.set_yscale("log"); b.set_ylabel("% of pixels"); b.set_xlabel("predicted defect probability"); b.grid(alpha=0.25)
plt.tight_layout(); plt.savefig(OUT / "fig_reliability.png"); plt.show()

In [ ]:
seg = ROOT / "results" / "segmentation"
def per_class(arch):
    runs = [json.load(open(seg / f"{arch}_resnet34_seed{s}_test_metrics.json"))["per_class"] for s in (0, 1, 2)]
    return {c: (np.mean([r[c]["IoU"] for r in runs]) * 100, np.std([r[c]["IoU"] for r in runs]) * 100, runs[0][c]["n"]) for c in runs[0]}
u, d = per_class("unet"), per_class("deeplabv3plus")
order = sorted(u, key=lambda c: u[c][0])
y = np.arange(len(order))
fig, ax = plt.subplots(figsize=(9, 9.5))
ax.barh(y + 0.2, [u[c][0] for c in order], 0.4, xerr=[u[c][1] for c in order], color=BLUE, label="U-Net (finalised)")
ax.barh(y - 0.2, [d[c][0] for c in order], 0.4, xerr=[d[c][1] for c in order], color="#B5BCC2", label="DeepLabV3+")
ax.set_yticks(y, [f"{c} (n={u[c][2]})" for c in order]); ax.set_xlim(0, 100)
ax.set_xlabel("test IoU (%), mean ± std over 3 seeds"); ax.legend(frameon=False, loc="lower right"); ax.grid(axis="x", alpha=0.25)
ax.set_title("Per-class segmentation IoU")
plt.tight_layout(); plt.savefig(OUT / "fig_per_class_iou.png"); plt.show()
print("saved to", OUT)